In [4]:

from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\User\Desktop\EduInsight-AI")
DATA_DIR = PROJECT_DIR / "data"

# Check and fix folders
for folder_name in ["raw", "processed"]:
    folder = DATA_DIR / folder_name

    if folder.is_file():
        old_name = DATA_DIR / f"{folder_name}_old"
        folder.rename(old_name)
        print(f"Existing file renamed: {old_name}")

    folder.mkdir(parents=True, exist_ok=True)

    print(f"{folder_name}: directory created =", folder.is_dir())

print("\nFolder structure fixed successfully!")


raw: directory created = True
Existing file renamed: C:\Users\User\Desktop\EduInsight-AI\data\processed_old
processed: directory created = True

Folder structure fixed successfully!


In [2]:
pip install nltk

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 3.9 MB/s eta 0:00:01
   ------------------------------------ --- 1.6/1.7 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 4.5 MB/s eta 0:00:00

   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ------------- -------------------------- 1/3 [regex]
   ------------- -------------------------- 1/3 [regex]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ----------


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:

# ============================================================
# EduInsight AI
# Phase 5: NLP Text Preprocessing
# ============================================================

from pathlib import Path
import pandas as pd
import re
import html
import nltk
import warnings

warnings.filterwarnings("ignore")

# 1. PROJECT PATHS

PROJECT_DIR = Path(r"C:\Users\User\Desktop\EduInsight-AI")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 2. LOAD CLEANED DATASETS

data_files = list(PROCESSED_DIR.glob("*_cleaned.csv"))

if not data_files:
    raise FileNotFoundError(
        "No cleaned CSV files found. Run Data Cleaning first."
    )

frames = []

for file in data_files:
    df_part = pd.read_csv(file)
    df_part["source_file"] = file.name
    frames.append(df_part)

df = pd.concat(frames, ignore_index=True)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# 3. IDENTIFY TEXT COLUMNS

text_columns = []

for col in df.select_dtypes(include=["object"]).columns:
    if col == "source_file":
        continue

    average_length = df[col].dropna().astype(str).str.len().mean()

    if pd.notna(average_length) and average_length > 20:
        text_columns.append(col)

print("\nDetected text columns:", text_columns)

if not text_columns:
    raise ValueError(
        "No long text columns detected in the dataset."
    )

# 4. TEXT CLEANING FUNCTION

def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Decode HTML entities
    text = html.unescape(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove email addresses
    text = re.sub(
        r"\b[\w.-]+@[\w.-]+\.\w+\b",
        " ",
        text
    )

    # Keep letters, numbers and spaces
    text = re.sub(r"[^\w\s]", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

# 5. APPLY TEXT CLEANING

for col in text_columns:
    df[col + "_clean"] = df[col].apply(clean_text)

# 6. TOKENIZATION

def tokenize_text(text):
    return text.split()

for col in text_columns:
    clean_col = col + "_clean"
    df[clean_col + "_tokens"] = df[clean_col].apply(tokenize_text)

# 7. TEXT STATISTICS

for col in text_columns:

    clean_col = col + "_clean"

    df[clean_col + "_word_count"] = (
        df[clean_col].apply(lambda x: len(x.split()))
    )

    print(f"\nStatistics for {col}:")
    print(
        "Average word count:",
        round(df[clean_col + "_word_count"].mean(), 2)
    )

# 8. DISPLAY RESULTS

print("\nOriginal and cleaned text:")

for col in text_columns:
    display(
        df[[col, col + "_clean"]].head(5)
    )

# 9. SAVE PREPROCESSED DATA

output_path = PROCESSED_DIR / "preprocessed_education_feedback.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 50)
print("TEXT PREPROCESSING COMPLETED")
print("=" * 50)
print("Saved to:", output_path)


Dataset shape: (54117, 51)
Columns: ['type', 'source', 'id', 'rating', 'num_token', 'sample_criteria', 'review_text', 'rating_bin', 'rating_cat', 'length_bin', 'source_file', 'df_name', 'count', '%', 'df_name.1', 'length_bin.1', 'count.1', '%.1', 'rating_cat.1', 'rating_bin.1', 'unnamed:_0', 'index', 'department', 'teacher_name', 'token_list', 'entry_idx', 'course_code', 'course_id', 'num_ratings', 'useful', 'easy', 'liked', 'num_reviews', 'rating_category', 'student_id', 'date', 'rating_facilities', 'rating_clubs', 'rating_careerservice', 'rating_internet', 'batch_id', 'course_review', 'course_review_(%)', 'teacher_review', 'teacher_review_(%)', 'university_review', 'university_review_(%)', 'total', 'total_(%)', 'sentence', 'overall']

Detected text columns: ['review_text', 'token_list', 'course_id', 'sentence']

Statistics for review_text:
Average word count: 36.22

Statistics for token_list:
Average word count: 26.57

Statistics for course_id:
Average word count: 1.02

Statistics fo

,review_text,review_text_clean
0,Dr. Zhong is a really good chemistry teacher. ...,dr zhong is a really good chemistry teacher sh...
1,I have her for behav. neuro and she is terribl...,i have her for behav neuro and she is terrible...
2,the prof. is real crap. it is a shame that the...,the prof is real crap it is a shame that the u...
3,He is a great guy and is pretty funny in class...,he is a great guy and is pretty funny in class...
4,"Plain and simple, this guy is HORRIBLE! He not...",plain and simple this guy is horrible he not o...


,token_list,token_list_clean
0,NaN,
1,NaN,
2,NaN,
3,NaN,
4,NaN,


,course_id,course_id_clean
0,NaN,
1,NaN,
2,NaN,
3,NaN,
4,NaN,


,sentence,sentence_clean
0,NaN,
1,NaN,
2,NaN,
3,NaN,
4,NaN,



TEXT PREPROCESSING COMPLETED
Saved to: C:\Users\User\Desktop\EduInsight-AI\data\processed\preprocessed_education_feedback.csv


In [6]:

from pathlib import Path
import pandas as pd
import zipfile
import re
import html

# 1. PATHS

PROJECT_DIR = Path(r"C:\Users\User\Desktop\EduInsight-AI")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

for folder in [RAW_DIR, PROCESSED_DIR]:
    if folder.is_file():
        raise ValueError(f"{folder} is a file, not a directory.")
    folder.mkdir(parents=True, exist_ok=True)

# 2. EXTRACT ZIP FILES

for zip_path in RAW_DIR.rglob("*.zip"):
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(RAW_DIR)

# 3. FIND DATA FILES

extensions = [".csv", ".xlsx", ".xls", ".json"]

data_files = [
    f for f in RAW_DIR.rglob("*")
    if f.is_file() and f.suffix.lower() in extensions
]

print("Files found:", len(data_files))

# 4. LOAD AND CLEAN

def clean_text(value):
    if pd.isna(value):
        return value

    value = html.unescape(str(value))
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"http\S+|www\S+", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()

for file in data_files:

    try:
        if file.suffix.lower() == ".csv":
            df = pd.read_csv(file)

        elif file.suffix.lower() in [".xlsx", ".xls"]:
            df = pd.read_excel(file)

        else:
            df = pd.read_json(file)

        df.columns = [
            str(c).strip().lower().replace(" ", "_")
            for c in df.columns
        ]

        df = df.dropna(how="all")
        df = df.dropna(axis=1, how="all")
        df = df.drop_duplicates()

        for col in df.select_dtypes(include="object").columns:
            df[col] = df[col].apply(clean_text)

        output = PROCESSED_DIR / f"{file.stem}_cleaned.csv"

        df.to_csv(output, index=False, encoding="utf-8-sig")

        print("Saved:", output.name, "| Shape:", df.shape)

    except Exception as error:
        print("Could not process:", file.name, "|", error)

print("\nDATA CLEANING FINISHED!")


Files found: 214
Could not process: ASQEJsonFile_(N=100_batch=1)_(2024-11-19_09-21-12)_(FULL_checked)_(V2).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=10)_(2024-11-21_23-08-33)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=11)_(2024-11-22_11-27-55)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=12)_(2024-11-22_11-27-55)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=13)_(2024-11-22_12-24-24)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=14)_(2024-11-23_10-41-02)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=15)_(2024-11-23_10-41-02)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batch=16)_(2024-11-23_12-11-09)_(FULL_checked).json | unhashable type: 'list'
Could not process: ASQEJsonFile_(N=100_batc